# 04 — Per-sample ResolVI correction for the reusable Visium HD pipeline

Run this notebook in the **pinned ResolVI/scvi-tools GPU environment** after
notebook 03.

For every selected sample it:

1. loads the minimally filtered, integer-count Proseg cell object;
2. validates the spatial coordinates and raw-count matrix;
3. prepares the ResolVI spatial-neighbor graph;
4. trains one ResolVI model per sample;
5. writes a resumable prepared-neighbor checkpoint and durable model;
6. calculates the ResolVI latent representation and posterior mixture
   proportions;
7. writes a cell-level H5AD retaining the transferred 2-µm annotations.

The model is intentionally fit **per sample**. The samples are independent
slides/individual tumors, and ResolVI models spatial contamination within each
slide rather than serving as the cross-sample integration model.

## Environment convention

This workflow preserves the API convention validated in the earlier project:

```python
CUDA_VISIBLE_DEVICES = one physical GPU
SCVI_DEVICE_SPEC = 1
model.train(..., accelerator="gpu", device=1)
```

Do not replace singular `device=1` with plural `devices=1` in this pinned
Pyro-backed ResolVI environment.

The notebook defaults to a one-sample smoke test. After that succeeds, change:

```python
SMOKE_TEST_ONLY = False
```


In [1]:
# FIRST CELL AFTER KERNEL RESTART — before importing torch/scvi
import os
import sys

GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Python:", sys.executable)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])


Python: /home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/bin/python
CUDA_VISIBLE_DEVICES: 0


In [2]:
# ---------------------------------------------------------------------
# Imports, configuration, and source manifest
# ---------------------------------------------------------------------
from __future__ import annotations

import gc
import hashlib
import inspect
import json
import os
import sys
import shutil
import time
import traceback
import warnings
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import pyro
import scipy.sparse as sp
import torch
from packaging.version import Version

import scvi
from scvi.external import RESOLVI

CONFIG_PATH = Path(
    os.environ.get(
        "VISIUMHD_PIPELINE_CONFIG",
        "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/"
        "derived_files/tbio8111_stardist_proseg_resolvi_v1/"
        "00_config/pipeline_config.json",
    )
)
CONFIG = json.loads(CONFIG_PATH.read_text())
CUSTOM_FUNCTION_DIR = Path(CONFIG["paths"]["functiondirs"])
if CUSTOM_FUNCTION_DIR.exists():
    if str(CUSTOM_FUNCTION_DIR) not in sys.path:
        sys.path.insert(0, str(CUSTOM_FUNCTION_DIR))
    print("Custom function directory enabled:", CUSTOM_FUNCTION_DIR)
else:
    warnings.warn(
        f"Configured functiondirs path does not exist: {CUSTOM_FUNCTION_DIR}. "
        "The built-in notebook helpers will be used."
    )
DERIVED_ROOT = Path(CONFIG["paths"]["derived_root"])
TEMP_ROOT = Path(CONFIG["paths"]["temp_root"])

QC_TRANSFER_ROOT = DERIVED_ROOT / "03_cell_qc_obs_transfer"
SOURCE_MANIFEST_PATH = QC_TRANSFER_ROOT / "resolvi_source_manifest.csv"
if not SOURCE_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "Run notebook 03 successfully first: " + str(SOURCE_MANIFEST_PATH)
    )
source_manifest = pd.read_csv(SOURCE_MANIFEST_PATH)
if source_manifest.empty:
    raise RuntimeError("The ResolVI source manifest is empty.")

RESOLVI_ROOT = DERIVED_ROOT / "04_resolvi"
RESOLVI_TEMP_ROOT = TEMP_ROOT / "04_resolvi"
RESOLVI_ROOT.mkdir(parents=True, exist_ok=True)
RESOLVI_TEMP_ROOT.mkdir(parents=True, exist_ok=True)

SAMPLE_IDS = source_manifest["sample"].astype(str).tolist()
SMOKE_TEST_ONLY = False
SMOKE_TEST_SAMPLE = SAMPLE_IDS[0]
SECTION_NAMES = [SMOKE_TEST_SAMPLE] if SMOKE_TEST_ONLY else SAMPLE_IDS

TRAIN_ACCELERATOR = "gpu"
SCVI_DEVICE_SPEC = 1  # one visible GPU; do not use 0
N_SPATIAL_NEIGHBORS = 10
MAX_EPOCHS = 50
TRAIN_BATCH_SIZE = 256
OOM_FALLBACK_BATCH_SIZE = 128
POSTERIOR_BATCH_SIZE = 512
POSTERIOR_NUM_SAMPLES = 3
POSTERIOR_SUMMARY_FREQUENCY = 50

MODEL_KWARGS = {
    "n_hidden": 32,
    "n_latent": 10,
    "n_layers": 2,
    "dropout_rate": 0.05,
    "dispersion": "gene",
    "gene_likelihood": "nb",
}

REUSE_PREPARED_NEIGHBORS = True
USE_EXISTING_FINAL = True
OVERWRITE_PREPARED = False
OVERWRITE_MODEL = False
CONTINUE_ON_ERROR = True
STRIP_RESOLVI_INTERNALS_FROM_FINAL = True
H5AD_COMPRESSION = "lzf"
RANDOM_SEED = 0
PIPELINE_VERSION = "reusable-resolvi-per-sample-v1"

print("scvi-tools:", scvi.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible CUDA devices:", torch.cuda.device_count())
print("RESOLVI.train signature:", inspect.signature(RESOLVI.train))
print("Samples selected:", SECTION_NAMES)

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch does not see a CUDA GPU.")
if torch.cuda.device_count() != 1:
    raise RuntimeError(
        "Expected exactly one visible GPU. Set GPU_ID in the first cell, "
        "restart the kernel, and rerun."
    )
if SCVI_DEVICE_SPEC == 0:
    raise ValueError(
        "SCVI_DEVICE_SPEC=0 becomes Lightning devices=0 in this API. "
        "Use 1 for one visible GPU."
    )
if Version(scvi.__version__) >= Version("1.6.0"):
    warnings.warn(
        "This notebook targets the pinned scvi.external.RESOLVI API used in "
        "the existing project. ResolVI moved to scVIVA Tools in newer "
        "releases; do not silently migrate environments mid-study."
    )

torch.set_float32_matmul_precision("high")
scvi.settings.seed = RANDOM_SEED
np.random.seed(RANDOM_SEED)


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Seed set to 0


Custom function directory enabled: /host_root/nethome/reny28/Projects/Custom_functions/python_functions
scvi-tools: 1.5.0.post1
torch: 2.11.0+cu130
CUDA available: True
Visible CUDA devices: 1
RESOLVI.train signature: (self, max_epochs: 'int' = 50, lr: 'float' = 0.003, lr_extra: 'float' = 0.01, extra_lr_parameters: 'tuple' = ('per_neighbor_diffusion_map', 'u_prior_means'), batch_size: 'int' = 512, weight_decay: 'float' = 0.0, eps: 'float' = 0.0001, n_steps_kl_warmup: 'int | None' = None, n_epochs_kl_warmup: 'int | None' = 20, plan_kwargs: 'dict | None' = None, expose_params: 'list' = (), **kwargs)
Samples selected: ['C2D15_14_60', 'C2D15_18_68', 'C2D15_22_24', 'C2D15_30_81', 'C2D15_7_93', 'Screen_14_60', 'Screen_18_68', 'Screen_22_24', 'Screen_30_81', 'Screen_7_93']


In [3]:
# ---------------------------------------------------------------------
# Paths, signatures, and atomic writers
# ---------------------------------------------------------------------
source_lookup = source_manifest.set_index("sample")


def paths_for_sample(sample: str) -> dict[str, Path]:
    row = source_lookup.loc[sample]
    durable = RESOLVI_ROOT / sample
    temporary = RESOLVI_TEMP_ROOT / sample
    durable.mkdir(parents=True, exist_ok=True)
    temporary.mkdir(parents=True, exist_ok=True)
    return {
        "sample": sample,
        "filtered": Path(row["resolvi_input_h5ad"]),
        "annotated": Path(row["annotated_h5ad"]),
        "prepared": temporary / f"{sample}_resolvi_prepared.h5ad",
        "prepared_signature": (
            temporary / f"{sample}_resolvi_prepared_signature.json"
        ),
        "model": durable / "model",
        "model_signature": durable / "model_signature.json",
        "history": durable / "training_history",
        "final": durable / f"{sample}_resolvi_annotated.h5ad",
        "summary": durable / f"{sample}_resolvi_summary.json",
        "success": durable / f"{sample}_resolvi_SUCCESS.json",
        "failure": durable / f"{sample}_resolvi_FAILURE.json",
    }


def atomic_json(payload, path: Path):
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, indent=2, default=str))
    temp.replace(path)


def sanitize_none(value):
    if value is None:
        return ""
    if isinstance(value, dict):
        return {str(k): sanitize_none(v) for k, v in value.items()}
    if isinstance(value, list):
        return [sanitize_none(v) for v in value]
    if isinstance(value, tuple):
        return tuple(sanitize_none(v) for v in value)
    return value


def atomic_write_h5ad(adata_obj: ad.AnnData, path: Path):
    temp = path.with_name(path.stem + ".tmp.h5ad")
    temp.unlink(missing_ok=True)
    adata_obj.uns = sanitize_none(dict(adata_obj.uns))
    adata_obj.write_h5ad(temp, compression=H5AD_COMPRESSION)
    check = ad.read_h5ad(temp, backed="r")
    if check.shape != adata_obj.shape:
        check.file.close()
        raise ValueError("H5AD read-back shape mismatch.")
    check.file.close()
    temp.replace(path)


def complete_model(path: Path) -> bool:
    return path.exists() and (path / "model.pt").exists()


def source_signature(path: Path) -> dict:
    return {
        "path": str(path.resolve()),
        "size": int(path.stat().st_size),
        "mtime_ns": int(path.stat().st_mtime_ns),
        "n_spatial_neighbors": int(N_SPATIAL_NEIGHBORS),
        "pipeline_version": PIPELINE_VERSION,
    }


def signature_hash(payload: dict) -> str:
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True).encode()
    ).hexdigest()


In [4]:
# ---------------------------------------------------------------------
# Count, coordinate, and prepared-neighbor validation
# ---------------------------------------------------------------------
def ensure_integer_counts(adata_obj: ad.AnnData, tolerance=1e-6):
    X = sp.csr_matrix(adata_obj.X)
    X.sum_duplicates()
    X.eliminate_zeros()
    X.sort_indices()

    if X.data.size:
        if not np.isfinite(X.data).all():
            raise ValueError("ResolVI input counts contain NaN or infinity.")
        if X.data.min() < 0:
            raise ValueError("ResolVI input counts contain negative values.")
        error = float(
            np.max(np.abs(X.data - np.rint(X.data)))
        )
        if error > tolerance:
            raise ValueError(
                f"ResolVI input is not integer-like; max deviation={error}."
            )
        X.data = np.rint(X.data).astype(np.int32, copy=False)

    totals = np.asarray(X.sum(axis=1)).reshape(-1)
    if np.any(totals <= 0):
        raise ValueError(
            f"{int((totals <= 0).sum())} ResolVI input cells have zero counts."
        )
    adata_obj.X = X


def prepare_spatial_coordinates(adata_obj: ad.AnnData) -> str:
    if "spatial" not in adata_obj.obsm:
        raise KeyError("ResolVI input lacks obsm['spatial'].")
    coordinates = np.asarray(
        adata_obj.obsm["spatial"],
        dtype=np.float32,
    )
    if coordinates.shape != (adata_obj.n_obs, 2):
        raise ValueError(
            f"Expected spatial shape {(adata_obj.n_obs, 2)}, "
            f"found {coordinates.shape}."
        )
    if not np.isfinite(coordinates).all():
        raise ValueError("Spatial coordinates contain nonfinite values.")
    adata_obj.obsm["X_spatial"] = coordinates
    return "obsm['spatial'] copied to obsm['X_spatial']"


def validate_prepared(adata_obj: ad.AnnData) -> dict:
    for key in ["index_neighbor", "distance_neighbor"]:
        if key not in adata_obj.obsm:
            raise KeyError(f"Prepared object lacks obsm[{key!r}].")

    indices = np.asarray(adata_obj.obsm["index_neighbor"])
    distances = np.asarray(adata_obj.obsm["distance_neighbor"])
    expected = (adata_obj.n_obs, N_SPATIAL_NEIGHBORS)

    if indices.shape != expected or distances.shape != expected:
        raise ValueError(
            f"Prepared neighbor shapes {indices.shape}/{distances.shape}; "
            f"expected {expected}."
        )
    if indices.size and (
        int(indices.min()) < 0
        or int(indices.max()) >= adata_obj.n_obs
    ):
        raise ValueError("Prepared neighbor indices are out of range.")
    if not np.isfinite(distances).all():
        raise ValueError("Prepared neighbor distances contain nonfinite values.")

    return {
        "index_neighbor_shape": list(map(int, indices.shape)),
        "distance_neighbor_shape": list(map(int, distances.shape)),
        "distance_min": float(distances.min()),
        "distance_median": float(np.median(distances)),
        "distance_max": float(distances.max()),
    }


def register_prepared_resolvi_adata(adata_obj: ad.AnnData):
    RESOLVI.setup_anndata(
        adata_obj,
        layer=None,
        batch_key=None,
        labels_key=None,
        prepare_data=False,
    )


In [5]:
# ---------------------------------------------------------------------
# Training and posterior helpers
# ---------------------------------------------------------------------
def save_history(model, directory: Path):
    directory.mkdir(parents=True, exist_ok=True)
    history = getattr(model, "history", None)
    if history is None:
        return

    if isinstance(history, dict):
        for key, value in history.items():
            try:
                pd.DataFrame(value).to_csv(directory / f"{key}.csv")
            except Exception as exc:
                warnings.warn(f"Could not save training history {key}: {exc}")
    else:
        try:
            pd.DataFrame(history).to_csv(directory / "history.csv")
        except Exception as exc:
            warnings.warn(f"Could not save training history: {exc}")


def train_model(model, batch_size: int):
    model.train(
        max_epochs=int(MAX_EPOCHS),
        batch_size=int(batch_size),
        accelerator=str(TRAIN_ACCELERATOR),
        device=SCVI_DEVICE_SPEC,
        enable_progress_bar=True,
        logger=False,
    )


def mixture_proportions(model, n_cells: int) -> np.ndarray:
    try:
        samples = model.sample_posterior(
            model=model.module.model_residuals,
            return_sites=["mixture_proportions"],
            summary_fun={"post_sample_means": np.mean},
            num_samples=int(POSTERIOR_NUM_SAMPLES),
            summary_frequency=int(POSTERIOR_SUMMARY_FREQUENCY),
        )
    except RuntimeError as exc:
        if "out of memory" not in str(exc).lower():
            raise
        warnings.warn(
            "Posterior mixture proportions hit CUDA OOM; retrying with "
            "one posterior sample."
        )
        torch.cuda.empty_cache()
        samples = model.sample_posterior(
            model=model.module.model_residuals,
            return_sites=["mixture_proportions"],
            summary_fun={"post_sample_means": np.mean},
            num_samples=1,
            summary_frequency=max(
                int(POSTERIOR_SUMMARY_FREQUENCY),
                100,
            ),
        )

    frame = pd.DataFrame(samples).T
    values = np.squeeze(
        np.asarray(
            frame.loc["post_sample_means", "mixture_proportions"]
        )
    )
    if values.shape == (3, n_cells):
        values = values.T
    if values.shape != (n_cells, 3):
        raise ValueError(
            f"Unexpected mixture_proportions shape {values.shape}; "
            f"expected {(n_cells, 3)}."
        )
    return values.astype(np.float32, copy=False)


In [6]:
# ---------------------------------------------------------------------
# Process one sample
# ---------------------------------------------------------------------
def process_sample(sample: str) -> dict:
    p = paths_for_sample(sample)

    if not p["filtered"].exists():
        raise FileNotFoundError(p["filtered"])

    run_signature = {
        **source_signature(p["filtered"]),
        "model_kwargs": MODEL_KWARGS,
        "max_epochs": int(MAX_EPOCHS),
        "train_batch_size": int(TRAIN_BATCH_SIZE),
        "posterior_batch_size": int(POSTERIOR_BATCH_SIZE),
        "posterior_num_samples": int(POSTERIOR_NUM_SAMPLES),
    }
    run_signature["signature_hash"] = signature_hash(run_signature)

    model_signature = {
        **source_signature(p["filtered"]),
        "model_kwargs": MODEL_KWARGS,
        "max_epochs": int(MAX_EPOCHS),
        "train_batch_size_requested": int(TRAIN_BATCH_SIZE),
        "oom_fallback_batch_size": int(OOM_FALLBACK_BATCH_SIZE),
        "pipeline_version": PIPELINE_VERSION,
    }
    model_signature["signature_hash"] = signature_hash(model_signature)

    if (
        USE_EXISTING_FINAL
        and p["final"].exists()
        and p["summary"].exists()
        and p["success"].exists()
    ):
        existing = json.loads(p["summary"].read_text())
        if existing.get("signature_hash") == run_signature["signature_hash"]:
            print("Reusing parameter-matched final ResolVI output:", sample)
            return existing
        print("Existing final ResolVI output has a different signature; rerunning.")

    started = time.time()
    prepared_reused = False
    model_reused = False
    training_batch_size_used = None

    pyro.clear_param_store()

    prepared_signature_matches = False
    if p["prepared_signature"].exists():
        previous_prepared_signature = json.loads(
            p["prepared_signature"].read_text()
        )
        prepared_signature_matches = (
            previous_prepared_signature.get("signature_hash")
            == signature_hash(source_signature(p["filtered"]))
        )

    if (
        REUSE_PREPARED_NEIGHBORS
        and p["prepared"].exists()
        and prepared_signature_matches
        and not OVERWRITE_PREPARED
    ):
        adata_obj = ad.read_h5ad(p["prepared"])
        ensure_integer_counts(adata_obj)
        prepare_spatial_coordinates(adata_obj)
        validation = validate_prepared(adata_obj)
        prepared_reused = True
        print("Reusing prepared spatial-neighbor checkpoint:", p["prepared"])
    else:
        adata_obj = ad.read_h5ad(p["filtered"])
        ensure_integer_counts(adata_obj)
        coordinate_source = prepare_spatial_coordinates(adata_obj)
        adata_obj.obs["sample"] = sample

        pyro.clear_param_store()
        RESOLVI.setup_anndata(
            adata_obj,
            layer=None,
            batch_key=None,
            labels_key=None,
            prepare_data=True,
            prepare_data_kwargs={
                "n_neighbors": int(N_SPATIAL_NEIGHBORS),
                "spatial_rep": "X_spatial",
            },
        )
        validation = validate_prepared(adata_obj)
        adata_obj.uns["resolvi_preparation"] = {
            "sample": sample,
            "coordinate_source": coordinate_source,
            "n_spatial_neighbors": int(N_SPATIAL_NEIGHBORS),
            "source_h5ad": str(p["filtered"]),
            "pipeline_version": PIPELINE_VERSION,
        }
        atomic_write_h5ad(adata_obj, p["prepared"])
        prepared_payload = source_signature(p["filtered"])
        prepared_payload["signature_hash"] = signature_hash(prepared_payload)
        atomic_json(prepared_payload, p["prepared_signature"])
        print("Saved prepared spatial-neighbor checkpoint:", p["prepared"])

    if prepared_reused:
        register_prepared_resolvi_adata(adata_obj)

    model_signature_matches = False
    if p["model_signature"].exists():
        existing_model_signature = json.loads(
            p["model_signature"].read_text()
        )
        model_signature_matches = (
            existing_model_signature.get("signature_hash")
            == model_signature["signature_hash"]
        )

    if (
        complete_model(p["model"])
        and model_signature_matches
        and not OVERWRITE_MODEL
    ):
        model = RESOLVI.load(
            str(p["model"]),
            adata=adata_obj,
            accelerator=TRAIN_ACCELERATOR,
            device=SCVI_DEVICE_SPEC,
        )
        model_reused = True
        print("Reusing parameter-matched trained model:", p["model"])
    else:
        if complete_model(p["model"]) and not model_signature_matches:
            if not OVERWRITE_MODEL:
                raise RuntimeError(
                    "A complete ResolVI model exists, but its saved signature "
                    "does not match the current source/configuration. Set "
                    "OVERWRITE_MODEL=True to retrain deliberately."
                )

        if p["model"].exists() and OVERWRITE_MODEL:
            shutil.rmtree(p["model"])
        if p["model_signature"].exists() and OVERWRITE_MODEL:
            p["model_signature"].unlink()

        pyro.clear_param_store()
        model = RESOLVI(adata_obj, **MODEL_KWARGS)
        print(model)

        try:
            train_model(model, TRAIN_BATCH_SIZE)
            training_batch_size_used = int(TRAIN_BATCH_SIZE)
        except RuntimeError as exc:
            if (
                "out of memory" not in str(exc).lower()
                or int(TRAIN_BATCH_SIZE)
                <= int(OOM_FALLBACK_BATCH_SIZE)
            ):
                raise

            warnings.warn(
                f"Training CUDA OOM at batch size {TRAIN_BATCH_SIZE}; "
                f"retraining from scratch with {OOM_FALLBACK_BATCH_SIZE}."
            )
            del model
            pyro.clear_param_store()
            torch.cuda.empty_cache()
            gc.collect()

            model = RESOLVI(adata_obj, **MODEL_KWARGS)
            train_model(model, OOM_FALLBACK_BATCH_SIZE)
            training_batch_size_used = int(OOM_FALLBACK_BATCH_SIZE)

        model.save(
            str(p["model"]),
            overwrite=True,
            save_anndata=False,
        )
        atomic_json(model_signature, p["model_signature"])
        save_history(model, p["history"])
        print("Saved model:", p["model"])

    adata_obj.obsm["X_resolvi"] = model.get_latent_representation(
        adata=adata_obj,
        batch_size=int(POSTERIOR_BATCH_SIZE),
    ).astype(np.float32)

    proportions = mixture_proportions(model, adata_obj.n_obs)
    adata_obj.obs[
        [
            "resolvi_true_proportion",
            "resolvi_diffusion_proportion",
            "resolvi_background_proportion",
        ]
    ] = proportions

    run_info = {
        **run_signature,
        "sample": sample,
        "completed": True,
        "prepared_h5ad": str(p["prepared"]),
        "model_dir": str(p["model"]),
        "final_h5ad": str(p["final"]),
        "scvi_tools_version": str(scvi.__version__),
        "torch_version": str(torch.__version__),
        "gpu_id_environment": GPU_ID,
        "train_accelerator": TRAIN_ACCELERATOR,
        "scvi_device_spec": SCVI_DEVICE_SPEC,
        "n_cells": int(adata_obj.n_obs),
        "n_genes": int(adata_obj.n_vars),
        "n_spatial_neighbors": int(N_SPATIAL_NEIGHBORS),
        "training_batch_size_used": (
            training_batch_size_used
            if training_batch_size_used is not None
            else "reused_model"
        ),
        "model_reused": bool(model_reused),
        "model_signature_hash": model_signature["signature_hash"],
        "prepared_neighbors_reused": bool(prepared_reused),
        "prepared_validation": validation,
        "runtime_minutes": float((time.time() - started) / 60.0),
        "pipeline_version": PIPELINE_VERSION,
    }
    adata_obj.uns["resolvi_run"] = sanitize_none(run_info)

    if STRIP_RESOLVI_INTERNALS_FROM_FINAL:
        for key in ["index_neighbor", "distance_neighbor"]:
            adata_obj.obsm.pop(key, None)
        internal_columns = [
            column
            for column in adata_obj.obs
            if column.startswith("_scvi") or column == "_indices"
        ]
        if internal_columns:
            adata_obj.obs.drop(
                columns=internal_columns,
                inplace=True,
            )
        for key in ["_scvi_uuid", "_scvi_manager_uuid"]:
            adata_obj.uns.pop(key, None)

    atomic_write_h5ad(adata_obj, p["final"])
    atomic_json(run_info, p["summary"])
    atomic_json(
        {
            "sample": sample,
            "completed": True,
            "final_h5ad": str(p["final"]),
            "model_dir": str(p["model"]),
            "signature_hash": run_signature["signature_hash"],
        },
        p["success"],
    )
    p["failure"].unlink(missing_ok=True)

    del model, adata_obj, proportions
    pyro.clear_param_store()
    gc.collect()
    torch.cuda.empty_cache()
    return run_info


In [ ]:
# ---------------------------------------------------------------------
# Run selected samples sequentially
# ---------------------------------------------------------------------
results = {}
failures = {}

for sample in SECTION_NAMES:
    print("\n" + "=" * 90)
    print("ResolVI sample:", sample)
    try:
        results[sample] = process_sample(sample)
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        failures[sample] = error
        p = paths_for_sample(sample)
        atomic_json(
            {"sample": sample, "error": error},
            p["failure"],
        )
        traceback.print_exc(limit=25)
        if not CONTINUE_ON_ERROR:
            raise
    finally:
        pyro.clear_param_store()
        gc.collect()
        torch.cuda.empty_cache()

summary = pd.DataFrame(results.values())
summary.to_csv(
    RESOLVI_ROOT / "all_samples_resolvi_summary.csv",
    index=False,
)
(
    RESOLVI_ROOT / "all_samples_resolvi_failures.json"
).write_text(json.dumps(failures, indent=2))

print("Completed:", sorted(results))
print("Failures:", json.dumps(failures, indent=2))

if failures:
    raise RuntimeError(
        "At least one ResolVI sample failed. Completed samples remain reusable."
    )



ResolVI sample: C2D15_14_60
Reusing parameter-matched final ResolVI output: C2D15_14_60

ResolVI sample: C2D15_18_68
Preparing data for training. This may take a while. RAPIDS SingleCell will be used if installed.
RAPIDS SingleCell is installed and can be imported
INFO     Generating sequential column names                                                                        
INFO     Generating sequential column names                                                                        


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/scvi/data/fields/_dataframe_field.py:187: UserWarning: Category 0 in adata.obs['_scvi_ind_x'] has fewer than 3 cells. Models may not train properly.
  categorical_mapping = _make_column_categorical(
... storing 'sample' as categorical


Saved prepared spatial-neighbor checkpoint: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8111/tmp/tbio8111_stardist_proseg_resolvi_v1/04_resolvi/C2D15_18_68/C2D15_18_68_resolvi_prepared.h5ad


/tmp/ipykernel_939/1466487206.py:144: FutureWarning: RESOLVI is a spatial transcriptomics model that will be moved to the scvi-tools spatial companion package `scviva-tools` starting in scvi-tools v1.5 and will no longer be supported here. It will be deprecated from scvi-tools in v1.6.
  model = RESOLVI(adata_obj, **MODEL_KWARGS)


RESOLVI Model with the following params: 
n_hidden: 32 n_latent: 10, n_layers: 2, dropout_rate: 0.05, dispersion: gene, gene_likelihood: nb n_neighbors: 10
Training status: Not Trained

...

Epoch 1/50:   0%|          | 0/50 [00:00<?, ?it/s]

2026-09-08 20:53:23 | [INFO] Guessed max_plate_nesting = 2


Epoch 50/50: 100%|██████████| 50/50 [06:02<00:00,  7.26s/it, elbo_train=4.37e+7]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 50/50: 100%|██████████| 50/50 [06:02<00:00,  7.26s/it, elbo_train=4.37e+7]
Saved model: /stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/derived_files/tbio8111_stardist_proseg_resolvi_v1/04_resolvi/Screen_7_93/model
Sampling local variables, batch:  28%|██▊       | 32/113 [00:03<00:09,  8.66it/s]